## Correlation Coefficient Based on Log Returns

Due to the large dataset of 7,000 stocks, finding effectivepairs can be computationally intensive. This notebook aims to make the process manageable and efficient. First, I am going to imputate missing data as pre-filter stocks based on sector/industry. Stocks within the same industry/sector are more likely to exhibit co-movement and correaltion due to market conditions and underlying drivers. 

This notebook will also use a correlation filter to eliminate pairs with low correlation in price movement. It will calculate the correlation matrix of stocks' historical returns. After collecting the correlation coefficients, it will select pairs with correlation coefficient above a certain threshold. The graphs below will show the transformation of data:

Here is what an individual csv file looks like:
| Date       | Open   | High   | Low    | Close  | Volume  | OpenInt |
|------------|--------|--------|--------|--------|---------|---------|
| 1999-11-22 | 27.886 | 29.702 | 27.044 | 29.702 | 27.002  | 0       |
| 1999-11-23 | 28.688 | 29.446 | 27.002 | 27.002 | 27.002  | 0       |
| 1999-11-24 | 27.083 | 28.012 | 27.002 | 27.717 | 57.502  | 0       |
| 1999-11-26 | 27.594 | 28.012 | 27.509 | 27.508 | 27.509  | 0       |
| 1999-11-29 | 27.676 | 28.650 | 27.380 | 28.432 | 27.388  | 0       |

Using the close price, load all 7000+ stocks into one CSV file:
| Date       | Stock A | Stock B | Stock C | Stock D | Stock E |
|------------|---------|---------|---------|---------|---------|
| 2023-11-01 | 100     | 102     | 105     | 110     | 98      |
| 2023-11-02 | 101     | 103     | 107     | 109     | 99      |
| 2023-11-03 | 103     | 104     | 106     | 111     | 97      |
| 2023-11-04 | 104     | 106     | 108     | 112     | 96      |
| 2023-11-05 | 105     | 108     | 109     | 113     | 95      |

Transform the daily price into daily return in percentage. The rationale behind daily return: percentage returns rather than prices would standardize the differences. e.g. A price change of $10 would represent a larger change for a $10 stock compared to a $100 stock. Each cell is represented in daily percentage change. The daily return in percentage for each stock uses the formula $$\text{Return} = \frac{\text{Price}_{today} - \text{Price}_{yesterday}}{\text{Price}_{yesterday}} \ times 100$$ 
| Date       | Stock A | Stock B | Stock C | Stock D | Stock E |
|------------|---------|---------|---------|---------|---------|
| 2023-11-02 | 1.00    | 0.98    | 1.90    | -0.91   | 1.02    |
| 2023-11-03 | 1.98    | 0.97    | -0.93   | 1.83    | -2.02   |
| 2023-11-04 | 0.97    | 1.92    | 1.89    | 0.90    | -1.03   |
| 2023-11-05 | 0.96    | 1.89    | 0.93    | 0.89    | -1.04   |


Once we have the daily returns, we calculate the correlation matrix:
|          | Stock A | Stock B | Stock C | Stock D | Stock E |
|----------|---------|---------|---------|---------|---------|
| Stock A  | 1.00    | 0.95    | 0.92    | 0.30    | 0.20    |
| Stock B  | 0.95    | 1.00    | 0.88    | 0.25    | 0.15    |
| Stock C  | 0.92    | 0.88    | 1.00    | 0.35    | 0.25    |
| Stock D  | 0.30    | 0.25    | 0.35    | 1.00    | 0.10    |
| Stock E  | 0.20    | 0.15    | 0.25    | 0.10    | 1.00    |

Then, using a correlation filter (i.e. 0.8), strongly correlated pairs will look like the following:
| Stock 1 | Stock 2 | Correlation |
|---------|---------|-------------|
| Stock A | Stock B | 0.95        |
| Stock A | Stock C | 0.92        |
| Stock B | Stock C | 0.88        |

In [5]:
## Import Libraries
import os
import pandas as pd
import yfinance as yf
import time

### Categorize Each Sector

Here is the list of sectors:
- Basic Material
- Communicaton Services
- Consumer Cyclical
- Consumer Defensive
- Energy
- Financial Services
- Healthcare
- Industrials
- Real Estate
- Technology
- Unknown
- Utilities

In [ ]:
source_dir = "dataset/stock_csv"
target_dir = "dataset/sector"
os.makedirs(target_dir, exist_ok=True)

def get_sector(ticker, retries=3, delay=5):
    for attempt in range(retries):
        try:
            stock_info = yf.Ticker(ticker).info
            return stock_info.get("sector")
        except Exception as e:
            print(f"Error fetching sector for {ticker}: {e}. Retrying ({attempt + 1}/{retries}) after delay...")
            time.sleep(delay)  # Wait before retrying
    print(f"Skipping {ticker} due to repeated errors.")
    return None

for stock_file in os.listdir(source_dir):
    if stock_file.endswith(".csv"):
        ticker = stock_file.split(".us")[0]  # Extract ticker before ".us"
        sector = get_sector(ticker)
        
        if sector is None:
            continue
        
        sector_path = os.path.join(target_dir, sector)
        os.makedirs(sector_path, exist_ok=True)
        os.rename(os.path.join(source_dir, stock_file), os.path.join(sector_path, stock_file))

print("Stocks have been organized by sector, skipping any invalid tickers.")


404 Client Error: Not Found for url: https://query2.finance.yahoo.com/v10/finance/quoteSummary/MITT_A?modules=financialData%2CquoteType%2CdefaultKeyStatistics%2CassetProfile%2CsummaryDetail&corsDomain=finance.yahoo.com&formatted=false&symbol=MITT_A&crumb=Ht%2FmROoUGM2
404 Client Error: Not Found for url: https://query2.finance.yahoo.com/v10/finance/quoteSummary/UBA?modules=financialData%2CquoteType%2CdefaultKeyStatistics%2CassetProfile%2CsummaryDetail&corsDomain=finance.yahoo.com&formatted=false&symbol=UBA&crumb=Ht%2FmROoUGM2
404 Client Error: Not Found for url: https://query2.finance.yahoo.com/v10/finance/quoteSummary/GCV_B?modules=financialData%2CquoteType%2CdefaultKeyStatistics%2CassetProfile%2CsummaryDetail&corsDomain=finance.yahoo.com&formatted=false&symbol=GCV_B&crumb=Ht%2FmROoUGM2
404 Client Error: Not Found for url: https://query2.finance.yahoo.com/v10/finance/quoteSummary/BBT_E?modules=financialData%2CquoteType%2CdefaultKeyStatistics%2CassetProfile%2CsummaryDetail&corsDomain=f

Error fetching sector for cald: Expecting value: line 1 column 1 (char 0). Retrying (1/3) after delay...


429 Client Error: Too Many Requests for url: https://query2.finance.yahoo.com/v10/finance/quoteSummary/CALD?modules=financialData%2CquoteType%2CdefaultKeyStatistics%2CassetProfile%2CsummaryDetail&corsDomain=finance.yahoo.com&formatted=false&symbol=CALD&crumb=Edge%3A+Too+Many+Requests


Error fetching sector for cald: Expecting value: line 1 column 1 (char 0). Retrying (2/3) after delay...


429 Client Error: Too Many Requests for url: https://query2.finance.yahoo.com/v10/finance/quoteSummary/CALD?modules=financialData%2CquoteType%2CdefaultKeyStatistics%2CassetProfile%2CsummaryDetail&corsDomain=finance.yahoo.com&formatted=false&symbol=CALD&crumb=Edge%3A+Too+Many+Requests


...

Error fetching sector for pbh: Expecting value: line 1 column 1 (char 0). Retrying (3/3) after delay...
Skipping pbh due to repeated errors.


429 Client Error: Too Many Requests for url: https://query2.finance.yahoo.com/v10/finance/quoteSummary/RIO?modules=financialData%2CquoteType%2CdefaultKeyStatistics%2CassetProfile%2CsummaryDetail&corsDomain=finance.yahoo.com&formatted=false&symbol=RIO&crumb=Edge%3A+Too+Many+Requests


Error fetching sector for rio: Expecting value: line 1 column 1 (char 0). Retrying (1/3) after delay...
Error fetching sector for rio: Expecting value: line 1 column 1 (char 0). Retrying (2/3) after delay...
Error fetching sector for rio: Expecting value: line 1 column 1 (char 0). Retrying (3/3) after delay...
Skipping rio due to repeated errors.


404 Client Error: Not Found for url: https://query2.finance.yahoo.com/v10/finance/quoteSummary/STAG_C?modules=financialData%2CquoteType%2CdefaultKeyStatistics%2CassetProfile%2CsummaryDetail&corsDomain=finance.yahoo.com&formatted=false&symbol=STAG_C&crumb=zOtyt%2FRhGzb
404 Client Error: Not Found for url: https://query2.finance.yahoo.com/v10/finance/quoteSummary/MICTW?modules=financialData%2CquoteType%2CdefaultKeyStatistics%2CassetProfile%2CsummaryDetail&corsDomain=finance.yahoo.com&formatted=false&symbol=MICTW&crumb=zOtyt%2FRhGzb
404 Client Error: Not Found for url: https://query2.finance.yahoo.com/v10/finance/quoteSummary/LOV?modules=financialData%2CquoteType%2CdefaultKeyStatistics%2CassetProfile%2CsummaryDetail&corsDomain=finance.yahoo.com&formatted=false&symbol=LOV&crumb=zOtyt%2FRhGzb
404 Client Error: Not Found for url: https://query2.finance.yahoo.com/v10/finance/quoteSummary/CYRXW?modules=financialData%2CquoteType%2CdefaultKeyStatistics%2CassetProfile%2CsummaryDetail&corsDomain=f

KeyboardInterrupt: 

In [5]:
data_directory = 'dataset/stock_csv'
output_directory = 'dataset'  # Directory where the final CSV will be saved
output_file = os.path.join(output_directory, 'all_stocks_closing_prices.csv')

collective_table = pd.DataFrame()

for filename in os.listdir(data_directory):
    if filename.endswith('.csv'):
        file_path = os.path.join(data_directory, filename)
        
        # Load the stock data with only 'Date' and 'Close' columns
        stock_data = pd.read_csv(file_path, usecols=['Date', 'Close'])
        
        # Extract the stock name from the filename (without the .csv extension)
        stock_name = filename.split('.')[0]
        
        # Rename the 'Close' column to the stock's name
        stock_data.rename(columns={'Close': stock_name}, inplace=True)
        
        # Set 'Date' as the index
        stock_data.set_index('Date', inplace=True)
        
        # Join the stock's closing prices to the main DataFrame
        if collective_table.empty:
            collective_table = stock_data  # Initialize with the first stock data
        else:
            collective_table = collective_table.join(stock_data, how='outer')  # Outer join to include all dates

# Reset the index to have Date as a column and sort by Date
collective_table.reset_index(inplace=True)
collective_table.sort_values(by='Date', inplace=True)

# Save to CSV in the specified output directory
collective_table.to_csv(output_file, index=False)

